# 01 - Digital Twin Fundamentals

## Introduction to Digital Twins

Welcome to the first module of the DualTwin Technologies 360° learning path. In this notebook, you'll learn the foundational concepts of digital twins through interactive examples and visualizations.

### Learning Objectives

By the end of this notebook, you will be able to:

- Define what a digital twin is in both technical and non-technical terms
- Explain why digital twins are valuable for organizations
- Identify where digital twins are used across industries
- Understand who interacts with digital twins and their different perspectives
- Describe how digital twins work at a high level

## What is a Digital Twin?

### Non-Technical Explanation

Imagine you have a detailed, living model of your car on your phone. This model knows everything about your actual car—how fast it's going, how much fuel it has, the engine temperature, and even when parts might need replacement. When something changes in your real car, the model updates automatically.

This virtual copy is a **digital twin**.

### Technical Definition

A **Digital Twin** is a dynamic, virtual representation of a physical entity (asset, process, or system) that is synchronized with its real-world counterpart through continuous data exchange.

In [ ]:
# First, let's import the libraries we'll need
import sys
sys.path.insert(0, '../../..')

import random
from datetime import datetime, timedelta
from dataclasses import dataclass, field
from typing import Dict, List, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set up plotting style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = [10, 6]

## A Simple Digital Twin Example

Let's create a simple digital twin of a temperature sensor. This will demonstrate the core concept of mirroring a physical entity in software.

In [ ]:
@dataclass
class PhysicalSensor:
    """Represents a physical temperature sensor."""
    sensor_id: str
    location: str
    actual_temperature: float = 25.0
    
    def read(self) -> float:
        """Read the sensor with some measurement noise."""
        noise = random.gauss(0, 0.5)
        return self.actual_temperature + noise
    
    def simulate_change(self, delta: float) -> None:
        """Simulate a temperature change in the physical world."""
        self.actual_temperature += delta


@dataclass
class DigitalTwinSensor:
    """Digital twin of a temperature sensor."""
    twin_id: str
    physical_sensor_id: str
    location: str
    current_temperature: float = 0.0
    last_updated: datetime = field(default_factory=datetime.now)
    history: List[Dict] = field(default_factory=list)
    
    def synchronize(self, reading: float) -> None:
        """Update the twin with a new reading from the physical sensor."""
        self.current_temperature = reading
        self.last_updated = datetime.now()
        self.history.append({
            'timestamp': self.last_updated,
            'temperature': reading
        })
    
    def get_average(self, n: int = 10) -> float:
        """Get average of last n readings."""
        if not self.history:
            return 0.0
        recent = self.history[-n:]
        return sum(r['temperature'] for r in recent) / len(recent)

In [ ]:
# Create a physical sensor and its digital twin
physical = PhysicalSensor(
    sensor_id="TEMP-001",
    location="Factory Floor - Zone A",
    actual_temperature=22.0
)

twin = DigitalTwinSensor(
    twin_id="DT-TEMP-001",
    physical_sensor_id="TEMP-001",
    location="Factory Floor - Zone A"
)

print("Physical Sensor Created:")
print(f"  ID: {physical.sensor_id}")
print(f"  Location: {physical.location}")
print(f"  Actual Temperature: {physical.actual_temperature}°C")

print("\nDigital Twin Created:")
print(f"  ID: {twin.twin_id}")
print(f"  Linked to: {twin.physical_sensor_id}")

In [ ]:
# Simulate the synchronization process
print("Simulating sensor readings and twin synchronization...\n")

physical_temps = []
twin_temps = []

for i in range(50):
    # Simulate temperature changes in the physical world
    if i == 10:
        physical.simulate_change(5)  # Temperature rises
    elif i == 30:
        physical.simulate_change(-3)  # Temperature drops
    
    # Read from physical sensor
    reading = physical.read()
    physical_temps.append(physical.actual_temperature)
    
    # Synchronize the digital twin
    twin.synchronize(reading)
    twin_temps.append(twin.current_temperature)

print(f"Collected {len(twin.history)} readings")
print(f"Final physical temperature: {physical.actual_temperature:.2f}°C")
print(f"Final twin temperature: {twin.current_temperature:.2f}°C")
print(f"Average (last 10 readings): {twin.get_average(10):.2f}°C")

In [ ]:
# Visualize the synchronization
fig, ax = plt.subplots(figsize=(12, 6))

time_points = range(len(physical_temps))

ax.plot(time_points, physical_temps, 'b-', linewidth=2, label='Physical (Actual)', alpha=0.7)
ax.plot(time_points, twin_temps, 'r--', linewidth=2, label='Digital Twin (Measured)', alpha=0.7)

ax.axvline(x=10, color='green', linestyle=':', alpha=0.5, label='Temperature Rise Event')
ax.axvline(x=30, color='orange', linestyle=':', alpha=0.5, label='Temperature Drop Event')

ax.set_xlabel('Time (readings)', fontsize=12)
ax.set_ylabel('Temperature (°C)', fontsize=12)
ax.set_title('Digital Twin Synchronization with Physical Sensor', fontsize=14)
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Why Digital Twins Matter

Digital twins provide several key benefits:

1. **Real-time Visibility**: See what's happening with your assets right now
2. **Historical Analysis**: Understand trends and patterns over time
3. **Predictive Capabilities**: Anticipate problems before they occur
4. **What-if Analysis**: Test scenarios without affecting physical systems
5. **Remote Monitoring**: Monitor assets from anywhere in the world

In [ ]:
# Demonstrate analytics capabilities
df = pd.DataFrame(twin.history)
df['timestamp'] = pd.to_datetime(df['timestamp'])

print("Digital Twin Analytics")
print("=" * 40)
print(f"\nTemperature Statistics:")
print(f"  Mean: {df['temperature'].mean():.2f}°C")
print(f"  Std Dev: {df['temperature'].std():.2f}°C")
print(f"  Min: {df['temperature'].min():.2f}°C")
print(f"  Max: {df['temperature'].max():.2f}°C")

# Simple anomaly detection
mean = df['temperature'].mean()
std = df['temperature'].std()
anomalies = df[abs(df['temperature'] - mean) > 2 * std]
print(f"\nAnomalies detected (>2 std): {len(anomalies)}")

## Where Are Digital Twins Used?

Digital twins are used across many industries:

| Industry | Example Applications |
|----------|---------------------|
| Manufacturing | Production lines, CNC machines, quality control |
| Energy | Wind turbines, power plants, solar farms |
| Buildings | HVAC systems, elevators, smart buildings |
| Transportation | Aircraft, ships, vehicle fleets |
| Healthcare | Medical devices, patient monitoring |

## How Digital Twins Work

The basic architecture of a digital twin system:

```
Physical World          Data Layer              Digital Twin
┌─────────────┐        ┌──────────┐           ┌─────────────┐
│   Sensors   │───────>│   IoT    │──────────>│   Virtual   │
│   Assets    │        │ Gateway  │           │   Model     │
│   Systems   │<───────│          │<──────────│   Analytics │
└─────────────┘        └──────────┘           └─────────────┘
                                                    │
                                                    v
                                              ┌─────────────┐
                                              │ Dashboards  │
                                              │   Alerts    │
                                              │   Reports   │
                                              └─────────────┘
```

## Summary

In this introduction, we learned:

1. **What**: A digital twin is a virtual representation of a physical entity that stays synchronized with real-world data

2. **Why**: Digital twins provide visibility, enable analytics, support prediction, and allow safe experimentation

3. **Where**: Used across manufacturing, energy, buildings, transportation, healthcare, and more

4. **Who**: Operators, engineers, data scientists, and managers all benefit from digital twins

5. **How**: Data flows from physical sensors through IoT infrastructure to digital models that provide insights

### Next Steps

- Explore the `examples_01_digital_twin_fundamentals.ipynb` notebook for more detailed examples
- Complete the exercises in `exercises_01_digital_twin_fundamentals.ipynb`
- View the HTML demo for animated visualizations
- Move on to Concept 02: Dual-Twin Architecture